# Memory and Observations
Companion notebook for the **Memory and Observations** section of Chapter 7.

A simple rule: **separate run state, memory, and observations**. Mixing them into one free-form blob makes later nodes read state in a messy way. Below is the book's own example state shape, built and validated as real Python (a lightweight `TypedDict`, not just JSON in a markdown cell).

In [1]:
from typing import TypedDict


class RunStatus(TypedDict):
    step: str
    status: str
    retry_count: int


class Fact(TypedDict):
    key: str
    value: object
    source: str
    validated: bool


class Event(TypedDict):
    type: str
    tool: str
    result: str


class Observation(TypedDict):
    source: str
    kind: str
    data: dict


class AgentState(TypedDict):
    """run_status: short-term, this execution only.
    facts: validated things believed true -- promoted from observations deliberately.
    events: what happened, for audit/replay.
    observations: fresh external signals, not yet decided whether they become memory."""

    run_status: RunStatus
    facts: list[Fact]
    events: list[Event]
    observations: list[Observation]

The book's own example, constructed as a real (type-checkable) `AgentState` value:

In [2]:
state: AgentState = {
    "run_status": {
        "step": "check_inventory",
        "status": "running",
        "retry_count": 1,
    },
    "facts": [
        {
            "key": "customer_tier",
            "value": "gold",
            "source": "crm",
            "validated": True,
        }
    ],
    "events": [
        {
            "type": "tool_call",
            "tool": "inventory_api",
            "result": "out_of_stock",
        }
    ],
    "observations": [
        {
            "source": "inventory_api",
            "kind": "tool_output",
            "data": {"sku": "A12", "stock": 0},
        }
    ],
}

import json
print(json.dumps(state, indent=2))

{
  "run_status": {
    "step": "check_inventory",
    "status": "running",
    "retry_count": 1
  },
  "facts": [
    {
      "key": "customer_tier",
      "value": "gold",
      "source": "crm",
      "validated": true
    }
  ],
  "events": [
    {
      "type": "tool_call",
      "tool": "inventory_api",
      "result": "out_of_stock"
    }
  ],
  "observations": [
    {
      "source": "inventory_api",
      "kind": "tool_output",
      "data": {
        "sku": "A12",
        "stock": 0
      }
    }
  ]
}


Each field has one job: `facts` stores what you believe is true, `events` stores what happened, `run_status` stores where execution stands, and `observations` stores fresh signals **before** you decide whether they should become memory -- not every observation should. A temporary API timeout is useful for this run, but doesn't belong in long-term memory.

In [3]:
# A trivial illustration of that last rule: only a validated fact is safe to keep
# across runs. An observation, on its own, is not automatically promoted to memory.
def is_memory_worthy(fact: Fact) -> bool:
    return fact["validated"]


for fact in state["facts"]:
    verdict = "keep in memory" if is_memory_worthy(fact) else "run-scoped only"
    print(f"{fact['key']} = {fact['value']!r} (source: {fact['source']}) -> {verdict}")

customer_tier = 'gold' (source: crm) -> keep in memory
